In [25]:
#0-1　基本設定
import numpy as np
import pandas as pd
import pickle
import json
import os

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
#0-2　テストデータからサンプル入力を生成

# FE2データを読込
test = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Kaggle/home-credit-default-risk/加工データ/03FE2/test_FE2.zstd.parquet')

# モデルの特徴量名を取得
SAVE_DIR = "/content/drive/MyDrive/home_credit_models_gpu"
with open(os.path.join(SAVE_DIR, "lgbm_fold0.pkl"), "rb") as f:
    model = pickle.load(f)

feature_names = model.feature_name()
print(f"モデルの特徴量数: {len(feature_names)}")

モデルの特徴量数: 669


In [27]:
#1-1　サンプルリクエストの作成

# テストデータの1行目からサンプルを作成
sample_row = test.drop(columns=['SK_ID_CURR']).iloc[0]

# 特徴量辞書を作成（モデルの特徴量名に合わせる）
sample_features = {}
for name in feature_names:
    if name in sample_row.index:
        val = sample_row[name]
        # NaN/カテゴリ型の処理
        if pd.isna(val):
            sample_features[name] = 0
        elif isinstance(val, (np.integer, np.int64, np.int32, np.int16, np.int8)):
            sample_features[name] = int(val)
        elif isinstance(val, (np.floating, np.float64, np.float32)):
            sample_features[name] = float(val)
        else:
            sample_features[name] = 0  # カテゴリ型は0で代替
    else:
        sample_features[name] = 0

# JSON出力
sample_event = {"features": sample_features}
with open("sample_event.json", "w") as f:
    json.dump(sample_event, f, indent=2)

print(f"サンプルリクエスト作成完了: {len(sample_features)}特徴量")
print(f"ファイル: sample_event.json")

サンプルリクエスト作成完了: 669特徴量
ファイル: sample_event.json


In [28]:
#1-2　ローカルでの推論テスト

# モデルで直接推論して期待値を確認
input_array = np.array(
    [[sample_features.get(name, 0) for name in feature_names]],
    dtype=np.float64
)
prob = float(model.predict(input_array)[0])
threshold = 0.24
decision = "auto_approve" if prob < threshold else "manual_review"

print(f"\n=== ローカル推論テスト ===")
print(f"予測確率: {prob:.6f}")
print(f"閾値: {threshold}")
print(f"判定: {decision}")

#1-3　モデルファイルのサイズ確認

model_path = os.path.join(SAVE_DIR, "lgbm_fold0.pkl")
size_mb = os.path.getsize(model_path) / 1024**2
print(f"\nモデルファイルサイズ: {size_mb:.1f} MB")
print(f"※Lambdaの/tmpは512MBまで使用可能")


=== ローカル推論テスト ===
予測確率: 0.057771
閾値: 0.24
判定: auto_approve

モデルファイルサイズ: 8.8 MB
※Lambdaの/tmpは512MBまで使用可能


In [29]:
'''
=================================================================
AWSデプロイ手順
=================================================================

■ アーキテクチャ概要
  GitHub → GitHub Actions → ECR（Dockerイメージ） → Lambda → API Gateway
  pushするだけで自動ビルド・デプロイされるCI/CDパイプライン

■ 手順1: S3バケットの作成（モデル保管用）
  1. AWSコンソール → S3 → バケットを作成
  2. バケット名: my-home-credit-model-2026-tn
  3. リージョン: ap-northeast-1（東京）
  4. lgbm_fold0.pkl を models/ フォルダにアップロード

■ 手順2: ECRリポジトリの作成
  1. AWSコンソール → ECR → リポジトリを作成
  2. リポジトリ名: home-credit-predictor

■ 手順3: IAMユーザーの作成（GitHub Actions用）
  1. AWSコンソール → IAM → ユーザー → ユーザーの作成
  2. ユーザー名: github-actions-user
  3. 以下のポリシーをアタッチ:
     - AmazonEC2ContainerRegistryPowerUser（ECRプッシュ用）
     - AWSLambda_FullAccess（Lambda更新用）
  4. アクセスキーを発行

■ 手順4: GitHubリポジトリにSecretsを登録
  1. リポジトリ → Settings → Secrets and variables → Actions
  2. 以下3つをRepository secretsとして登録:
     - AWS_ACCESS_KEY_ID（手順3で発行）
     - AWS_SECRET_ACCESS_KEY（手順3で発行）
     - AWS_ACCOUNT_ID

■ 手順5: リポジトリにデプロイファイルを追加
  以下のファイルをGitHubリポジトリに追加:

  deploy/Dockerfile:
    FROM public.ecr.aws/lambda/python:3.12
    RUN dnf install -y libgomp && dnf clean all
    RUN pip install --no-cache-dir --only-binary :all: \
        lightgbm==4.3.0 numpy scipy
    COPY lambda_function.py ${LAMBDA_TASK_ROOT}/
    CMD ["lambda_function.lambda_handler"]

  deploy/lambda_function.py:
    推論用Lambda関数（S3からモデル読込 → 予測 → JSON返却）

  .github/workflows/deploy.yml:
    pushトリガーでDockerビルド → ECRプッシュ → Lambda更新を自動実行

  ※ deploy/ 配下のファイルを変更してpushすると自動デプロイが走る

■ 手順6: Lambda関数の初回作成
  GitHub Actionsが初回実行時に自動でLambda関数を作成:
    - パッケージタイプ: コンテナイメージ
    - メモリ: 1024MB（モデル読込に必要）
    - タイムアウト: 60秒（コールドスタート対策）
    - 環境変数:
      MODEL_BUCKET = my-home-credit-model-2026-tn
      MODEL_KEY = models/lgbm_fold0.pkl
      THRESHOLD = 0.24
    - 実行ロールにS3読取権限（AmazonS3ReadOnlyAccess）を付与

■ 手順7: API Gatewayの作成（CloudShellで実行）
  1. REST API作成（API名: home-credit-api）
  2. /predict リソースを作成
  3. POSTメソッドを追加 → Lambda関数とプロキシ統合
  4. Lambdaに実行権限を付与
  5. prod ステージにデプロイ
  6. エンドポイントURLが発行される

■ テスト方法
  curl -X POST \
    https://{api-id}.execute-api.ap-northeast-1.amazonaws.com/prod/predict \
    -H "Content-Type: application/json" \
    -d '{"features": {"EXT_SOURCE_2": 0.5, "EXT_SOURCE_3": 0.3}}'

  レスポンス例:
    {
      "probability": 0.073,
      "threshold": 0.24,
      "decision": "auto_approve",
      "features_received": 2
    }

■ デプロイ時に遭遇した問題と解決策
  1. zipデプロイの250MB制限
     lightgbm + scipy + numpy の依存関係がLambdaのzip上限250MBを超過。
     → Docker（ECR）方式に切り替えて10GB上限で解決。

  2. NumPyのGCCバージョン要求
     Python 3.11ベースイメージ（Amazon Linux 2, GCC 7.3）では
     NumPy 2.x のビルドに必要な GCC >= 9.3 を満たせない。
     → Python 3.12ベースイメージ（Amazon Linux 2023）に変更し、
        --only-binary :all: でビルド済みwheelのみ使用。

  3. libgomp不足
     LightGBMが並列処理に必要とするlibgomp.so.1が
     Lambdaベースイメージに含まれていない。
     → Dockerfileに dnf install -y libgomp を追加。
'''


'\n=================================================================\nAWSデプロイ手順\n=================================================================\n\n■ アーキテクチャ概要\n  GitHub → GitHub Actions → ECR（Dockerイメージ） → Lambda → API Gateway\n  pushするだけで自動ビルド・デプロイされるCI/CDパイプライン\n\n■ 手順1: S3バケットの作成（モデル保管用）\n  1. AWSコンソール → S3 → バケットを作成\n  2. バケット名: my-home-credit-model-2026-tn\n  3. リージョン: ap-northeast-1（東京）\n  4. lgbm_fold0.pkl を models/ フォルダにアップロード\n\n■ 手順2: ECRリポジトリの作成\n  1. AWSコンソール → ECR → リポジトリを作成\n  2. リポジトリ名: home-credit-predictor\n\n■ 手順3: IAMユーザーの作成（GitHub Actions用）\n  1. AWSコンソール → IAM → ユーザー → ユーザーの作成\n  2. ユーザー名: github-actions-user\n  3. 以下のポリシーをアタッチ:\n     - AmazonEC2ContainerRegistryPowerUser（ECRプッシュ用）\n     - AWSLambda_FullAccess（Lambda更新用）\n  4. アクセスキーを発行\n\n■ 手順4: GitHubリポジトリにSecretsを登録\n  1. リポジトリ → Settings → Secrets and variables → Actions\n  2. 以下3つをRepository secretsとして登録:\n     - AWS_ACCESS_KEY_ID（手順3で発行）\n     - AWS_SECRET_ACCESS_KEY（手順3で発行）\n     - AWS_ACCOUNT_ID\n\n■ 手順5

In [30]:
#2-1　簡易テスト
import requests

url = "https://0roeepf6u2.execute-api.ap-northeast-1.amazonaws.com/prod/predict"
response = requests.post(url, json={"features": {"EXT_SOURCE_2": 0.5}})
print(response.json())

{'probability': 0.469379, 'threshold': 0.24, 'decision': 'manual_review', 'features_received': 1}


In [37]:
#2-2　テスト(自動承認、2次審査)

import json

# TARGET=0（正常返済）から1件
row_good = df[df["TARGET"] == 0].sample(1, random_state=42).drop(columns=["TARGET"]).iloc[0]
row_good = row_good[row_good.index.isin(df.select_dtypes(include="number").columns)]
features_good = {k: (None if pd.isna(v) else float(v)) for k, v in row_good.items()}

print("=" * 60)
print("テスト1: 正常返済の実レコード")
print("=" * 60)
response = requests.post(url, json={"features": features_good})
print(json.dumps(response.json(), indent=2))

# TARGET=1（デフォルト）から1件
row_bad = df[df["TARGET"] == 1].sample(1, random_state=42).drop(columns=["TARGET"]).iloc[0]
row_bad = row_bad[row_bad.index.isin(df.select_dtypes(include="number").columns)]
features_bad = {k: (None if pd.isna(v) else float(v)) for k, v in row_bad.items()}

print("\n" + "=" * 60)
print("テスト2: デフォルトの実レコード")
print("=" * 60)
response = requests.post(url, json={"features": features_bad})
print(json.dumps(response.json(), indent=2))

テスト1: 正常返済の実レコード
{
  "probability": 0.031548,
  "threshold": 0.24,
  "decision": "auto_approve",
  "features_received": 631
}

テスト2: デフォルトの実レコード
{
  "probability": 0.301442,
  "threshold": 0.24,
  "decision": "manual_review",
  "features_received": 631
}
